# 

# Notebook to run a comparison of models exported to ONNX files

## Caveats:
Models have not been retrained in this range - matters for OO which has an extended range
Currently runs on the whole set - not the training / test that might be different 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils
import optuna
import Evaluation
import shap

from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix, classification_report, accuracy_score, average_precision_score, log_loss, auc
from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance
import seaborn as sns
from sklearn.calibration import CalibrationDisplay
from matplotlib import cm


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

import onnx
import numpy as np
from onnx import helper, numpy_helper, TensorProto
import onnxruntime as ort
import Utils
import matplotlib.pyplot as plt
import importlib
import pandas as pd
import seaborn as sns
import Evaluation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df_OO = Utils.get_dataframe("Data/OOwmatchattempts.root", folder_name="DF_*")
df_PbPb = Utils.get_dataframe("Data/PbPbwmatchattempts.root", folder_name="DF_*")


In [ ]:
df_OO = Utils.process_dataframe(df_OO, makedummies=False)
df_PbPb = Utils.process_dataframe(df_PbPb, makedummies=False)

In [ ]:
df_OO = Utils.subsample(df_OO, frac = 0.3)
df_PbPb = Utils.subsample(df_PbPb, frac = 0.3)
# Subsample due to memory conditions, about a third for now

In [ ]:
FEATURES_OO = ['DeltaDirection', 'PullPt', 'APullPhi', 'PullTanl', 'PtMFT',
       'CPhiPhiMFT', 'DeltaR', 'SameSign', 'DeltaEta', 'DeltaTanl',
       'RelPtDiff', 'CYYMFT', 'C1Pt1PtMFT', 'PullPhi', 'CXXMFT',
       'C1PtPhiMFT', 'YMCH', 'XMCH', 'DeltaPt', 'ADeltaPhi', 'PullR',
       'PullX', 'PullY', 'CXYMFT', 'CTglTglMCH', 'DeltaPhi', 'C1PtXMFT']
FEATURES_PBPB = ['RelPtDiff', 'SameSign', 'PtMFT', 'PullPt', 'CPhiPhiMFT',
       'C1Pt1PtMFT', 'CTglTglMCH', 'CPhiPhiMCH', 'TanlMFT', 'CXXMFT',
       'CYYMFT', 'DeltaDirection', 'DeltaR', 'ADeltaPhi', 'etaMFT',
       'PullR', 'DeltaPt', 'C1PtPhiMFT', 'DeltaEta', 'ADeltaX',
       'APullPhi', 'DeltaTanl', 'CYYMCH', 'ADeltaY', 'CTglTglMFT',
       'PullTanl', 'CXXMCH', 'InvQPtMFT', 'PtMCH', 'C1Pt1PtMCH', 'PullY',
       'CXYMFT', 'APullX', 'DeltaX', 'APullY', 'DeltaPhi', 'C1PtXMFT',
       'DeltaY', 'CTglXMCH', 'etaMCH', 'PullX', 'YMCH', 'TanlMCH', 'XMCH',
       'CPhiXMFT', 'CTglXMFT', 'CPhiYMFT', 'C1PtYMFT', 'CTglPhiMFT',
       'PullPhi', 'XMFT', 'YMFT', 'CPhiYMCH', 'CTglYMFT', 'PhiMFT',
       'PhiMCH', 'C1PtTglMCH', 'CTglYMCH', 'C1PtTglMFT', 'CPhiXMCH']
MODEL_OO = "Models_Proper/model_OO_25features.onnx"
MODEL_PBPB = "Models_Proper/PbPbV1.onnx"


In [ ]:
sessOO = ort.InferenceSession(MODEL_OO)
input_name = sessOO.get_inputs()[0].name
df_OO['score'] = sessOO.run(
    None,
    {input_name: df_OO[FEATURES_OO].to_numpy(dtype=np.float32)}
)[0] #.ravel()# ravel used for the binary classifier, for the classifier we want the fuller array

In [ ]:
sessPbPb = ort.InferenceSession(MODEL_OO)
input_name = sessOO.get_inputs()[0].name
df_PbPb['score'] = sessPbPb.run(
    None,
    {input_name: df_PbPb[FEATURES_OO].to_numpy(dtype=np.float32)}
)[0] #.ravel()# ravel used for the binary classifier, for the classifier we want the fuller array

In [ ]:

sessPbPb = ort.InferenceSession(MODEL_PBPB)
input_name = sessOO.get_inputs()[0].name
df_PbPb['pbpbmodelonpbpbdata'] = sessPbPb.run(
    None,
    {input_name: df_PbPb[FEATURES_PBPB].to_numpy(dtype=np.float32)}
)[0]

In [ ]:
Utils.plot_metrics_vs_xy(df_PbPb,
    feature_x="PtMCH",
    fmin_x = 0.3,
    fmax_x = 2.5,
    feature_y="MatchAttempts",
    fmin_y = 0.0,
    fmax_y = 300.0,
    threshold=0.0,
    metrics_fn=Utils.inhousemetrics,
    x_bins=2,
    y_bins=10,
    metric_col_prefix = 'pbpbmodelonpbpbdata')
#TODO for presentation: crossection slices for pt & matchattempts respectively - metricwisefeatureplots1D for comparsion of models & systems evaluated on one another
#TODO: rerun the entire EDA notebook for the missing match cetegory for better categorization

In [ ]:
Utils.plot_metrics_vs_xy(df_PbPb,
    feature_x="PtMCH",
    fmin_x = 0.3,
    fmax_x = 2.5,
    feature_y="MatchAttempts",
    fmin_y = 0.0,
    fmax_y = 3000.0,
    threshold=0.6,
    metrics_fn=Utils.inhousemetrics,
    x_bins=2,
    y_bins=10,)

    #TODO: Add functionality for plotting & comparing the missing match % or categories
    #TODO: score disrtibution decomposition - e.g. momentum, pt, eta, matchattempts --- featurewise - depending on score distributions 
    # TODO: prepareo2physicss based JSON configuration for merger

In [ ]:
Utils.plot_metrics_vs_xy(df_OO,
    feature_x="PtMCH",
    fmin_x = 0.3,
    fmax_x = 2.5,
    feature_y="MatchAttempts",
    fmin_y = 0.0,
    fmax_y = 1000.0,
    threshold=0.9,
    metrics_fn=Utils.inhousemetrics,
    x_bins=2,
    y_bins=10,)